In [8]:
from xgboost import XGBClassifier

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <89AD948E-E564-3266-867D-7AF89D6488F0> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

In [6]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv("football_data_clean.csv")

# Ensure chronological order
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# Encode categorical features
label_encoders = {}
for col in ["home_team_name", "away_team_name.x", "league"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save encoders for future predictions

# Define features and target
features = [
    "home_team_name", "away_team_name.x", "league", "home_away_flag",
    "PPG_home_last_5_lag5", "PPG_away_last_5_lag5",
    "xG_home_team_lag5", "xG_away_team_lag5",
    "conversion_rate_home_lag5", "conversion_rate_away_lag5",
    "average_goals_per_match_pre_match", "average_corners_per_match_pre_match",
    "average_cards_per_match_pre_match", "home_team_shots", "away_team_shots",
    "home_team_shots_on_target", "away_team_shots_on_target",
    "home_team_fouls", "away_team_fouls", "home_team_possession", "away_team_possession",
    "home_team_yellow_cards", "away_team_yellow_cards",
    "home_team_red_cards", "away_team_red_cards", "Corners_home_team", "Corners_away_team",
    "Passes_home_team", "Passes_away_team", "accurate_passes_home_team", "Passes.accurate_away_team",
    "PPDA_home_team", "PPDA_away_team",
    "Match.tempo_home_team", "Match.tempo_away_team",
    "Average.pass.length_home_team", "Average.pass.length_away_team",
    "Shots.outside.PA_home_team", "Shots.outside.PA.on_target_home_team",
    "Positional.attacks_home_team", "Counterattacks_home_team",
    "Defensive.duels_won_home_team", "Defensive.duels_won_away_team"
]

X = df[features]
y = df["result"]

# Encode target variable (1 → 0, X → 1, 2 → 2)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Scale features
scaler = StandardScaler()
X[features] = scaler.fit_transform(X[features])

# **Sliding Window Training & Prediction**
initial_train_size = int(len(df) * 0.7)  # Start with 70% training data
window_size = 50  # How many matches to train on before predicting the next one

# Track predictions and actual results
predictions = []
actual_results = []

# Rolling training and prediction
for i in range(initial_train_size, len(df) - 1):
    # Define rolling window
    X_train, y_train = X.iloc[i - window_size:i], y[i - window_size:i]
    X_next, y_next = X.iloc[i:i+1], y[i:i+1]  # The next match to predict
    
    # Train the XGBoost model
    model = XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")
    model.fit(X_train, y_train)
    
    # Predict the next match
    y_pred = model.predict(X_next)
    
    # Store prediction and actual result
    predictions.append(y_pred[0])
    actual_results.append(y_next.iloc[0])

# Convert predictions back to match results (1/X/2)
predicted_results = label_encoder.inverse_transform(predictions)
actual_results = label_encoder.inverse_transform(actual_results)

# **Evaluate Performance**
accuracy = accuracy_score(actual_results, predicted_results)
print(f"\nRolling Window Model Accuracy: {accuracy:.2f}")

# Show last few predictions
for i in range(-10, 0):
    print(f"Predicted: {predicted_results[i]}, Actual: {actual_results[i]}")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <89AD948E-E564-3266-867D-7AF89D6488F0> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O